# Graph RAG Tutorial Part 3: LLM Integration and Graph Expansion

This notebook demonstrates advanced GraphRAG techniques:
1. Integrating with LLMs for response generation
2. Building an interactive graph expansion system
3. Implementing structural graph matching
4. Creating a complete GraphRAG pipeline

## Prerequisites

```bash
pip install chromadb sentence-transformers plotly openai
```

You'll also need an OpenAI API key (or you can use a local model).

## 1. Setup and Imports

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import re

# For RAG components
try:
    import chromadb
    from sentence_transformers import SentenceTransformer
    RAG_AVAILABLE = True
    print("ChromaDB and SentenceTransformers available")
except ImportError:
    RAG_AVAILABLE = False
    print("Warning: Install chromadb sentence-transformers for full functionality")

# For LLM integration
try:
    from openai import OpenAI
    LLM_AVAILABLE = True
    print("OpenAI available")
except ImportError:
    LLM_AVAILABLE = False
    print("Warning: Install openai for LLM integration")

## 2. Enhanced GraphRAG Class with Graph Matching

This class extends the basic GraphRAG with structural matching capabilities.

In [ ]:
class GraphRAG:
    """
    A Graph-based Retrieval-Augmented Generation system using topologic_fast.
    
    This class stores a knowledge base of graph fragments and supports retrieval
    using both semantic similarity and structural graph matching.
    """
    
    def __init__(self, fragments=None):
        """
        Initialize the GraphRAG instance.
        
        Parameters:
        - fragments: List of graph fragments as tuples (node_type, [neighbor_types])
        """
        if RAG_AVAILABLE:
            self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
            self.chroma_client = chromadb.Client()
        else:
            self.embedder = None
            self.chroma_client = None
        
        self.collection = None
        self.knowledge_graphs = []
        
        if fragments:
            self.knowledge_graphs = [self._fragment_to_graph(f) for f in fragments]
    
    def _fragment_to_graph(self, fragment):
        """
        Convert a fragment definition to a topologic_fast graph.
        
        Parameters:
        - fragment: List of tuples (node_type, [neighbor_types])
        
        Returns:
        - tf.Graph object
        """
        vertex_map = {}
        vertices = []
        edges = []
        
        # Create vertices for each node type
        for node, neighbors in fragment:
            if node not in vertex_map:
                # Position vertices in a line for simplicity
                v = tf.Vertex.ByCoordinates(len(vertex_map), 0, 0)
                # NOTE: topologic_fast doesn't have Dictionary yet
                # In a full implementation, we would store type as metadata
                vertex_map[node] = {'vertex': v, 'type': node}
                vertices.append(v)
        
        # Create edges
        edge_set = set()
        for node, neighbors in fragment:
            v1 = vertex_map[node]['vertex']
            for n in neighbors:
                if n in vertex_map:
                    v2 = vertex_map[n]['vertex']
                    # Avoid duplicate edges
                    edge_key = tuple(sorted([node, n]))
                    if edge_key not in edge_set:
                        edges.append(tf.Edge.ByStartVertexEndVertex(v1, v2))
                        edge_set.add(edge_key)
        
        return {
            'graph': tf.Graph.ByVerticesEdges(vertices, edges),
            'vertex_map': vertex_map,
            'fragment': fragment
        }
    
    def retrieve_by_structure(self, input_graph, top_k=3):
        """
        Retrieve top-k most structurally similar graphs from the knowledge base.
        
        Uses graph metrics for similarity comparison.
        
        Parameters:
        - input_graph: tf.Graph to compare
        - top_k: Number of results to return
        
        Returns:
        - List of (score, graph_info) tuples
        """
        input_order = input_graph.Order()
        input_size = input_graph.Size()
        input_density = input_graph.Density() if input_order > 1 else 0
        
        matches = []
        for kg_info in self.knowledge_graphs:
            kg = kg_info['graph']
            
            # Simple structural similarity based on graph metrics
            kg_order = kg.Order()
            kg_size = kg.Size()
            kg_density = kg.Density() if kg_order > 1 else 0
            
            # Compute similarity score (higher is more similar)
            order_sim = 1.0 / (1.0 + abs(input_order - kg_order))
            size_sim = 1.0 / (1.0 + abs(input_size - kg_size))
            density_sim = 1.0 - abs(input_density - kg_density)
            
            score = (order_sim + size_sim + density_sim) / 3.0
            matches.append((score, kg_info))
        
        # Sort by score descending
        matches.sort(reverse=True, key=lambda x: x[0])
        return matches[:top_k]
    
    def graph_to_prompt(self, graph_info):
        """
        Convert a graph fragment to a text prompt.
        
        Parameters:
        - graph_info: Dict with 'graph', 'vertex_map', 'fragment'
        
        Returns:
        - Text description of the graph structure
        """
        prompts = []
        for node, neighbors in graph_info['fragment']:
            if neighbors:
                prompts.append(f"A '{node}' connected to {', '.join(neighbors)}.")
            else:
                prompts.append(f"A '{node}' with no connections.")
        return "\n".join(prompts)

print("GraphRAG class defined")

## 3. Create Knowledge Base of Spatial Patterns

We'll define common architectural patterns as graph fragments.

In [ ]:
# Define common spatial patterns as graph fragments
# Each fragment is a list of (node_type, [connected_node_types])

known_patterns = [
    # Pattern 1: Bedroom suite
    [
        ("bedroom", ["bathroom", "closet"]),
        ("bathroom", ["bedroom"]),
        ("closet", ["bedroom"])
    ],
    
    # Pattern 2: Kitchen area
    [
        ("kitchen", ["diningroom", "pantry"]),
        ("diningroom", ["kitchen"]),
        ("pantry", ["kitchen"])
    ],
    
    # Pattern 3: Common area hub
    [
        ("livingroom", ["hallway"]),
        ("hallway", ["livingroom", "bedroom", "bathroom", "kitchen", "closet"]),
        ("bedroom", ["hallway"]),
        ("bathroom", ["hallway"]),
        ("kitchen", ["hallway"]),
        ("closet", ["hallway"])
    ],
    
    # Pattern 4: Office layout
    [
        ("reception", ["corridor"]),
        ("corridor", ["reception", "office", "conference", "breakroom"]),
        ("office", ["corridor"]),
        ("conference", ["corridor"]),
        ("breakroom", ["corridor"])
    ]
]

# Initialize GraphRAG with knowledge base
rag = GraphRAG(known_patterns)

print(f"Created GraphRAG with {len(rag.knowledge_graphs)} spatial patterns:")
for i, kg_info in enumerate(rag.knowledge_graphs):
    g = kg_info['graph']
    print(f"  Pattern {i+1}: {g.Order()} rooms, {g.Size()} connections")

## 4. Interactive Graph Expansion System

This system builds a spatial graph incrementally, using RAG to suggest additions.

In [ ]:
class GraphExpander:
    """
    Interactive system for expanding spatial graphs using RAG.
    """
    
    def __init__(self, rag, openai_client=None):
        """
        Initialize the graph expander.
        
        Parameters:
        - rag: GraphRAG instance with knowledge base
        - openai_client: Optional OpenAI client for LLM suggestions
        """
        self.rag = rag
        self.client = openai_client
        self.graph = None
        self.vertices = []
        self.edges = []
        self.room_types = []  # Store room types since we can't use Dictionary
        self.room_positions = {}  # Store positions for visualization
    
    def initialize(self, start_type="hallway"):
        """
        Initialize the graph with a starting room.
        
        Parameters:
        - start_type: Type of the initial room
        """
        v = tf.Vertex.ByCoordinates(0, 0, 0)
        self.vertices = [v]
        self.edges = []
        self.room_types = [start_type]
        self.room_positions = {0: (0, 0)}
        self.graph = tf.Graph.ByVerticesEdges(self.vertices, self.edges)
        
        return self.graph
    
    def get_suggestions(self, top_k=3):
        """
        Get suggestions for graph expansion based on current state.
        
        Returns:
        - List of text prompts describing potential additions
        """
        if not self.graph:
            return ["Initialize the graph first with initialize()"]
        
        matches = self.rag.retrieve_by_structure(self.graph, top_k)
        suggestions = []
        
        for score, kg_info in matches:
            prompt = self.rag.graph_to_prompt(kg_info)
            suggestions.append(f"[Score: {score:.2f}]\n{prompt}")
        
        return suggestions
    
    def add_room(self, room_type, connect_to_index=None):
        """
        Add a new room to the graph.
        
        Parameters:
        - room_type: Type of room to add
        - connect_to_index: Index of existing room to connect to (default: last added)
        
        Returns:
        - Updated graph
        """
        if connect_to_index is None:
            connect_to_index = len(self.vertices) - 1
        
        if connect_to_index < 0 or connect_to_index >= len(self.vertices):
            raise ValueError(f"Invalid connect_to_index: {connect_to_index}")
        
        # Calculate position for new vertex (simple grid layout)
        new_idx = len(self.vertices)
        x = (new_idx % 5) * 2
        y = (new_idx // 5) * 2
        
        # Create new vertex
        new_v = tf.Vertex.ByCoordinates(x, y, 0)
        self.vertices.append(new_v)
        self.room_types.append(room_type)
        self.room_positions[new_idx] = (x, y)
        
        # Create edge to connected room
        existing_v = self.vertices[connect_to_index]
        new_edge = tf.Edge.ByStartVertexEndVertex(new_v, existing_v)
        self.edges.append(new_edge)
        
        # Rebuild graph
        self.graph = tf.Graph.ByVerticesEdges(self.vertices, self.edges)
        
        return self.graph
    
    def get_state(self):
        """
        Get current graph state as text.
        """
        if not self.graph:
            return "Graph not initialized"
        
        lines = [f"Current graph: {self.graph.Order()} rooms, {self.graph.Size()} connections"]
        
        for i, room_type in enumerate(self.room_types):
            v = self.vertices[i]
            degree = self.graph.VertexDegree(v)
            lines.append(f"  [{i}] {room_type}: {degree} connection(s)")
        
        return "\n".join(lines)
    
    def suggest_with_llm(self, max_tokens=100):
        """
        Use LLM to suggest next room based on current state and RAG context.
        
        Returns:
        - Suggestion text or None if LLM not available
        """
        if not self.client:
            return None
        
        # Get RAG context
        suggestions = self.get_suggestions(top_k=2)
        context = "\n\n".join(suggestions)
        
        # Build prompt
        current_rooms = ", ".join(self.room_types)
        prompt = f"""You are an assistant that expands spatial graphs by suggesting new rooms.

Current rooms: {current_rooms}

Example patterns from knowledge base:
{context}

Suggest ONE new room to add. Respond with:
Add '<room_type>' connected to '<existing_room_type>'

Or respond with STOP if the layout is complete."""
        
        try:
            response = self.client.chat.completions.create(
                model="gpt-4",
                messages=[
                    {"role": "system", "content": "You suggest spatial layout expansions."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=max_tokens,
                temperature=0.3
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            return f"Error: {e}"

print("GraphExpander class defined")

## 5. Test Graph Expansion

In [ ]:
# Create expander (without LLM for now)
expander = GraphExpander(rag)

# Initialize with a hallway
expander.initialize("hallway")
print("Initial state:")
print(expander.get_state())

# Get suggestions
print("\nSuggestions from knowledge base:")
for i, suggestion in enumerate(expander.get_suggestions(top_k=2)):
    print(f"\n--- Suggestion {i+1} ---")
    print(suggestion)

In [ ]:
# Manually expand the graph based on suggestions
print("Expanding graph...\n")

# Add rooms connected to the hallway
rooms_to_add = [
    ("livingroom", 0),  # Connect to hallway (index 0)
    ("bedroom", 0),
    ("bathroom", 0),
    ("kitchen", 0),
    ("closet", 2),  # Connect to bedroom (index 2)
]

for room_type, connect_to in rooms_to_add:
    expander.add_room(room_type, connect_to)
    print(f"Added {room_type} connected to {expander.room_types[connect_to]}")

print("\n" + "=" * 50)
print(expander.get_state())

## 6. Visualize the Expanded Graph

In [ ]:
def visualize_expanded_graph(expander, title="Expanded Spatial Graph"):
    """
    Visualize the expanded graph with room types.
    """
    # Color scheme by room type
    color_map = {
        'hallway': '#A0A0A0',
        'livingroom': '#90EE90',
        'bedroom': '#87CEEB',
        'bathroom': '#E6E6FA',
        'kitchen': '#FFDAB9',
        'closet': '#D2B48C',
        'diningroom': '#F0E68C',
        'pantry': '#FFE4C4',
        'office': '#98FB98',
        'conference': '#FFD700',
        'reception': '#87CEEB',
        'breakroom': '#FFB6C1',
        'corridor': '#D3D3D3',
    }
    
    fig = go.Figure()
    
    graph = expander.graph
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Draw edges
    for edge in edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) >= 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color='gray', width=3),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw vertices
    for i, v in enumerate(vertices):
        coords = v.Coordinates()
        room_type = expander.room_types[i]
        color = color_map.get(room_type, '#CCCCCC')
        degree = graph.VertexDegree(v)
        
        fig.add_trace(go.Scatter(
            x=[coords[0]],
            y=[coords[1]],
            mode='markers+text',
            marker=dict(
                size=40,
                color=color,
                line=dict(color='black', width=2)
            ),
            text=[room_type],
            textposition='middle center',
            textfont=dict(size=9),
            name=room_type,
            hovertext=f"{room_type}<br>Connections: {degree}",
            hoverinfo='text'
        ))
    
    fig.update_layout(
        title=title,
        xaxis=dict(title='X', scaleanchor='y', scaleratio=1),
        yaxis=dict(title='Y'),
        width=800,
        height=600,
        showlegend=False
    )
    
    return fig

fig = visualize_expanded_graph(expander, "House Layout from Graph Expansion")
fig.show()

## 7. LLM Integration (Optional)

If you have an OpenAI API key, you can enable LLM-guided graph expansion.

In [ ]:
# Uncomment and run this cell if you have an OpenAI API key

# import getpass
# 
# if LLM_AVAILABLE:
#     print("Enter your OpenAI API key:")
#     api_key = getpass.getpass()
#     client = OpenAI(api_key=api_key)
#     
#     # Create expander with LLM
#     expander_llm = GraphExpander(rag, openai_client=client)
#     expander_llm.initialize("hallway")
#     
#     # Get LLM suggestion
#     suggestion = expander_llm.suggest_with_llm()
#     print(f"LLM Suggestion: {suggestion}")
# else:
#     print("OpenAI not available. Install with: pip install openai")

print("LLM integration example (uncomment to use)")

## 8. Complete GraphRAG Pipeline

Let's put together a complete pipeline that answers questions about a spatial layout.

In [ ]:
def answer_spatial_query(graph, room_types, query):
    """
    Answer a query about a spatial graph.
    
    This is a rule-based system. For LLM-based answers,
    integrate with OpenAI or another provider.
    """
    query_lower = query.lower()
    vertices = graph.Vertices()
    
    # Query: "How many rooms?"
    if "how many" in query_lower and "room" in query_lower:
        return f"There are {graph.Order()} rooms in the layout."
    
    # Query: "What is connected to X?"
    for i, room in enumerate(room_types):
        if room.lower() in query_lower and "connected" in query_lower:
            adjacent = graph.AdjacentVertices(vertices[i])
            adj_types = []
            for adj_v in adjacent:
                adj_coords = adj_v.Coordinates()
                for j, v in enumerate(vertices):
                    v_coords = v.Coordinates()
                    if (abs(adj_coords[0] - v_coords[0]) < 0.01 and 
                        abs(adj_coords[1] - v_coords[1]) < 0.01):
                        adj_types.append(room_types[j])
                        break
            if adj_types:
                return f"The {room} is connected to: {', '.join(adj_types)}."
            else:
                return f"The {room} has no connections."
    
    # Query: "Which room has most connections?"
    if "most connection" in query_lower:
        max_degree = 0
        max_room = None
        for i, v in enumerate(vertices):
            degree = graph.VertexDegree(v)
            if degree > max_degree:
                max_degree = degree
                max_room = room_types[i]
        return f"The {max_room} has the most connections ({max_degree})."
    
    # Query: "Is X connected to Y?"
    for i, room1 in enumerate(room_types):
        for j, room2 in enumerate(room_types):
            if i != j and room1.lower() in query_lower and room2.lower() in query_lower:
                dist = graph.Distance(vertices[i], vertices[j])
                if dist == 1:
                    return f"Yes, {room1} is directly connected to {room2}."
                elif dist > 1:
                    return f"No, {room1} is not directly connected to {room2}, but they are {dist} steps apart."
                else:
                    return f"{room1} and {room2} are not connected."
    
    return "I couldn't understand the query. Try asking about connections or room counts."

# Test queries
print("Spatial Query System")
print("=" * 50)

test_queries = [
    "How many rooms are there?",
    "What is connected to the hallway?",
    "Which room has the most connections?",
    "Is the bedroom connected to the kitchen?",
    "What is connected to the bedroom?"
]

for query in test_queries:
    answer = answer_spatial_query(expander.graph, expander.room_types, query)
    print(f"\nQ: {query}")
    print(f"A: {answer}")

## Summary

In this notebook, we covered advanced GraphRAG techniques:

1. **Enhanced GraphRAG Class** with structural graph matching
2. **Knowledge Base** of spatial patterns for retrieval
3. **Interactive Graph Expansion** using RAG suggestions
4. **LLM Integration** for intelligent suggestions (optional)
5. **Complete Pipeline** for answering spatial queries

### Key Insights

- **Structural matching** complements semantic search for graph retrieval
- **Pattern libraries** encode domain knowledge (architectural layouts)
- **Incremental expansion** allows interactive design exploration
- **Rule-based + LLM** hybrid approaches work well for spatial queries

### Limitations and Future Work

- **Dictionary support** - topologic_fast doesn't yet have Dictionary for metadata
  - NOTE: This feature is not yet implemented in topologic_fast
- **Graph matching** - Full subgraph isomorphism would improve retrieval
  - NOTE: Graph.Match is not yet implemented in topologic_fast
- **Visualization** - 3D layouts and floor plan generation

### Next Steps

Check out the RDF_BOT_Export and RDF_BOT_Import notebooks to learn about:
- Exporting graphs to RDF/BOT format
- Interoperability with semantic web tools
- Building information modeling (BIM) integration